# Task 1: a demonstration and acceptance script

This notebook contains **domain data, expected results and checks**, not implementations of the reasoners. The checks become runnable when your `engine.py` provides the starter API.

**Scope:** Task 1 only: one rule base, forward chaining, backward chaining, variables and recursion. Task 2 is separate.

In each section: predict the result, run the cell when the implementation is ready, compare, then explain why. Keep production engine code in `.py` modules. For final submission, put the domain data in `domains.py` and import it here.

### Run discipline

Run from the repository root, or ensure it is on Python's import path. Restart the kernel and Run All before accepting the demo. Restart after editing imported engine modules to avoid using stale code. No third-party library is needed by the engine or the checks below; Jupyter is the demonstration environment.


## 1. Agree the miniature world

The initial facts describe six parent relationships. Read `parent(x, y)` as “x is a parent of y”.

| Parent | Child |
|---|---|
| mary | ann |
| ann | sam |
| sam | leo |
| mary | bob |
| bob | ivy |
| tom | nina |

Mary's family has two branches. The path Mary → Ann → Sam → Leo forces recursive reasoning beyond one application. Tom and Nina form a disconnected family, giving useful negative checks.

Facts are ground tuples; patterns use variables starting with `?`.

**Predict:** Who are Mary's children, grandchildren and descendants? Does Mary have a relationship to Nina under these rules?


In [3]:
# Development fixture: domain data only, with no reasoning logic.
# Before submission, move these definitions into domains.py and import them here.
DEMO_FACTS = {
    ("parent", "mary", "ann"),
    ("parent", "ann", "sam"),
    ("parent", "sam", "leo"),
    ("parent", "mary", "bob"),
    ("parent", "bob", "ivy"),
    ("parent", "tom", "nina"),
}

DEMO_RULES = [
    {"if": [("parent", "?x", "?y"), ("parent", "?y", "?z")],
     "then": ("grandparent", "?x", "?z")},
    {"if": [("parent", "?x", "?y")],
     "then": ("ancestor", "?x", "?y")},
    {"if": [("parent", "?x", "?y"), ("ancestor", "?y", "?z")],
     "then": ("ancestor", "?x", "?z")},
]

for fact in sorted(DEMO_FACTS):
    print(fact)


('parent', 'ann', 'sam')
('parent', 'bob', 'ivy')
('parent', 'mary', 'ann')
('parent', 'mary', 'bob')
('parent', 'sam', 'leo')
('parent', 'tom', 'nina')


## 2. Agree what each rule means

1. **Grandparent:** two parent links conclude a grandparent relationship.
2. **Direct ancestry:** every parent is an ancestor.
3. **Recursive ancestry:** if x is a parent of y, and y is an ancestor of z, x is an ancestor of z.

These are inferences about relationships; firing them does not change who anyone's parent is.

**Key matching check:** in rule 1, `?y` must be the same person in both conditions. You cannot combine `parent(mary, ann)` with `parent(bob, ivy)` to conclude `grandparent(mary, ivy)`. That conclusion is true through a different pair: `parent(mary, bob)` and `parent(bob, ivy)`.

The engine must understand the tuple/dictionary structure without having special code for `parent`, `grandparent`, or `ancestor`.


In [ ]:
# Expected results are stated independently of either reasoner's output.
EXPECTED_GRANDPARENTS = {
    ("grandparent", "mary", "sam"),
    ("grandparent", "mary", "ivy"),
    ("grandparent", "ann", "leo"),
}
EXPECTED_ANCESTORS = {
    ("ancestor", "mary", "ann"),
    ("ancestor", "mary", "sam"),
    ("ancestor", "mary", "leo"),
    ("ancestor", "mary", "bob"),
    ("ancestor", "mary", "ivy"),
    ("ancestor", "ann", "sam"),
    ("ancestor", "ann", "leo"),
    ("ancestor", "sam", "leo"),
    ("ancestor", "bob", "ivy"),
    ("ancestor", "tom", "nina"),
}
EXPECTED_CLOSURE = DEMO_FACTS | EXPECTED_GRANDPARENTS | EXPECTED_ANCESTORS
assert len(EXPECTED_CLOSURE) == 19
print("Expected closure: 6 parent + 3 grandparent + 10 ancestor facts = 19.")


## 3. Connect the notebook to your engine

Keep the starter signatures:

- `forward_chain(wm, rules)` adds conclusions to WM and returns WM.
- `backward_chain(wm, rules, query, subst=None)` yields bindings for proofs and does not change WM.

The starter functions raise `NotImplementedError` until you implement them. An import failure here means the project modules are not yet in place or on the import path; it is not a reasoning failure.

The display helpers below only format and compare answers. They do not derive facts or prove goals. Bindings are resolved using the supplied `substitute` function because the raw answer may contain variable-to-variable chains.


In [1]:
from reasoning.engine import WorkingMemory, forward_chain, backward_chain, substitute

In [2]:

def show_facts(facts):
    for fact in sorted(facts):
        print(fact)


def backward_answers(wm, query):
    """Display-independent answer set, using the supplied substitution function."""
    return {substitute(query, binding)
            for binding in backward_chain(wm, DEMO_RULES, query)}


def explain_check(label, actual, expected):
    if actual != expected:
        print("Missing:", sorted(expected - actual))
        print("Unexpected:", sorted(actual - expected))
    assert actual == expected, label
    print("PASS:", label)


## 4. Show basic working-memory matching

**Action:** ask the original WM for `parent(mary, ?child)`.

**Expected:** Ann and Bob, and nobody else. Asking is matching existing facts; it does not apply production rules.

**Explain:** `?child` receives different bindings for different matches. A set has no guaranteed iteration order, so sort for presentation and compare answer sets.


In [4]:
wm_initial = WorkingMemory(DEMO_FACTS)
children = {substitute(("parent", "mary", "?child"), b)
            for b in wm_initial.ask(("parent", "mary", "?child"))}
explain_check("Mary's children", children,
              {("parent", "mary", "ann"), ("parent", "mary", "bob")})
show_facts(children)
assert not wm_initial.holds(("grandparent", "mary", "sam"))
print("Grandparent relationship is not an initial fact; it must be inferred.")


PASS: Mary's children
('parent', 'mary', 'ann')
('parent', 'mary', 'bob')
Grandparent relationship is not an initial fact; it must be inferred.


## 5. Forward chaining: derive the full closure

**Action:** create fresh WM, apply the three rules repeatedly, and print the new facts.

**Expected:** 13 new facts and 19 facts in total, exactly matching `EXPECTED_CLOSURE`.

Illustrative dependency sequence:

- Original parent facts support the three grandparent facts and six direct ancestor facts.
- Those ancestor facts support `ancestor(mary, sam)`, `ancestor(mary, ivy)` and `ancestor(ann, leo)`.
- The recursive rule also establishes `ancestor(mary, leo)`.
- Eventually a full pass adds nothing: the fixpoint.

This describes dependencies, not a required number of passes. Your engine may make new facts available immediately or only in the next pass. Both can produce the same correct closure.

**Explain:** the engine has to join all rule conditions under consistent bindings, substitute those bindings into the conclusion, and keep going until no new ground fact appears. Record at least one concrete firing in your explanation or optional trace.


In [5]:
wm_forward = WorkingMemory(DEMO_FACTS)
before_forward = wm_forward.snapshot()
forward_chain(wm_forward, DEMO_RULES)
new_facts = wm_forward.facts - before_forward
print("Derived facts:")
show_facts(new_facts)
print("Initial:", len(before_forward), "New:", len(new_facts),
      "Total:", len(wm_forward.facts))
explain_check("Exact forward closure", wm_forward.snapshot(), EXPECTED_CLOSURE)
assert before_forward <= wm_forward.facts
print("PASS: original facts are preserved (monotonic growth).")


NotImplementedError: 

## 6. Forward chaining: demonstrate termination

**Action:** run forward chaining again on the completed WM.

**Expected:** the second run finishes and adds nothing. Re-deriving an existing fact must not count as progress.

**Explain:** with this finite domain, ground facts and no generation of new constants, there are finitely many possible facts. The set prevents duplicates, and the loop stops when nothing new is added.


In [ ]:
closure_before_rerun = wm_forward.snapshot()
forward_chain(wm_forward, DEMO_RULES)
assert wm_forward.snapshot() == closure_before_rerun
print("PASS: second run adds no facts; fixpoint remains at 19 facts.")


## 7. Backward chaining: a true ground goal

**Action:** on a NEW WM containing only the six original parent facts, query `grandparent(mary, sam)`.

**Expected:** success, even though no grandparent fact is stored initially.

Explain the proof:

1. Match the query against the grandparent rule's conclusion.
2. Establish `parent(mary, ?middle)` and `parent(?middle, sam)`.
3. The first condition has two candidates: Ann and Bob.
4. Ann also satisfies the second condition; Bob does not.
5. At least one full proof means the goal succeeds.

**Critical:** do not feed backward chaining the forward closure. That could let it answer by looking up previously inferred facts and conceal a broken implementation.


In [ ]:
wm_backward = WorkingMemory(DEMO_FACTS)
backward_baseline = wm_backward.snapshot()
query = ("grandparent", "mary", "sam")
answers = backward_answers(wm_backward, query)
explain_check("True ground goal", answers, {query})
assert wm_backward.snapshot() == backward_baseline
print("PASS: backward proof leaves WM unchanged.")


## 8. Backward chaining: a false ground goal

**Action:** query `grandparent(mary, leo)`.

**Expected:** no proof, because Leo is three parent links from Mary. The same pair WILL satisfy ancestry, so this tests that the rule predicates are respected.

“No proof” means the query is not established by this rule base and these facts. The engine has not derived an explicit negative fact.


In [ ]:
query = ("grandparent", "mary", "leo")
explain_check("Unsupported grandparent goal", backward_answers(wm_backward, query), set())
print("The query is not established by the current facts and rules.")


## 9. Backward chaining: multiple answers

**Action:** query `grandparent(mary, ?who)`.

**Expected:** exactly Ivy and Sam. Returning only the first proof is insufficient for a query with multiple answers.

**Explain:** alternatives must be explored without allowing bindings from one failed branch to contaminate another.


In [ ]:
query = ("grandparent", "mary", "?who")
answers = backward_answers(wm_backward, query)
explain_check("All grandchildren", answers,
              {("grandparent", "mary", "ivy"), ("grandparent", "mary", "sam")})
show_facts(answers)


## 10. Backward chaining: a recursive proof

**Action:** query `ancestor(mary, leo)` on the original facts.

**Expected:** success through Mary → Ann → Sam → Leo.

Explain the successful proof branch:

1. To prove `ancestor(mary, leo)`, use a parent link to Ann and prove `ancestor(ann, leo)`.
2. Use Ann's parent link to Sam and prove `ancestor(sam, leo)`.
3. Use the direct ancestry rule and the stored fact `parent(sam, leo)`.

Each application of the recursive rule needs fresh variable names. Its `?x`, `?y` and `?z` must not accidentally share bindings with another application of the same rule. Freshening changes internal names, not the rule's meaning.


In [ ]:
query = ("ancestor", "mary", "leo")
explain_check("Recursive ancestor proof", backward_answers(wm_backward, query), {query})


## 11. Recursive variable queries and disconnected families

**Action:** ask for all Mary's descendants, then all Leo's ancestors, then an unsupported relationship to Nina.

**Expected:**

| Query | Answers |
|---|---|
| `ancestor(mary, ?who)` | ann, bob, ivy, leo, sam |
| `ancestor(?who, leo)` | ann, mary, sam |
| `ancestor(mary, nina)` | no proof |
| `ancestor(mary, mary)` | no proof in this acyclic fixture |

These ancestry rules describe paths of at least one parent link, not automatic self-ancestry.


In [ ]:
query = ("ancestor", "mary", "?who")
expected = {("ancestor", "mary", name) for name in ("ann", "bob", "ivy", "leo", "sam")}
answers = backward_answers(wm_backward, query)
explain_check("All Mary's descendants", answers, expected)
show_facts(answers)

query = ("ancestor", "?who", "leo")
expected = {("ancestor", name, "leo") for name in ("ann", "mary", "sam")}
explain_check("All Leo's ancestors", backward_answers(wm_backward, query), expected)

for query in [("ancestor", "mary", "nina"), ("ancestor", "mary", "mary")]:
    explain_check(str(query), backward_answers(wm_backward, query), set())
assert wm_backward.snapshot() == backward_baseline
print("PASS: all backward queries leave WM unchanged.")


## 12. Compare the two reasoning directions

**Action:** ask both reasoners for all ancestor pairs and all grandparent pairs.

**Expected:** identical answers for each predicate: ten ancestor pairs and three grandparent pairs.

Forward chaining starts with facts and derives conclusions. Backward chaining starts with a goal and seeks supporting facts. They use the SAME `DEMO_RULES`, and should agree on this bounded demonstration domain.

Agreement alone is not enough: the earlier checks against independently listed expected facts protect against both implementations making the same mistake.


In [ ]:
for predicate in ("ancestor", "grandparent"):
    query = (predicate, "?a", "?b")
    forward_answers = {substitute(query, b) for b in wm_forward.ask(query)}
    backward_result = backward_answers(wm_backward, query)
    explain_check(f"Forward/backward agreement: {predicate}",
                  backward_result, forward_answers)
    print("Answer count:", len(backward_result))


## 13. Demonstrate engine/knowledge separation

**Action:** add a new rule AS DATA, with no edit to the engine:

`IF grandparent(?a, ?b) THEN two_generations_above(?a, ?b)`

**Expected:** both reasoners support the new predicate; Mary is two generations above Sam. This is a simple demonstration that the generic engine is not hardcoded to the original predicates.

Use a separate rule list and fresh memories so this experiment does not alter the earlier fixtures.


In [ ]:
extended_rules = DEMO_RULES + [
    {"if": [("grandparent", "?a", "?b")],
     "then": ("two_generations_above", "?a", "?b")}
]
wm_extended_forward = WorkingMemory(DEMO_FACTS)
forward_chain(wm_extended_forward, extended_rules)
expected_new = {("two_generations_above", fact[1], fact[2])
                for fact in EXPECTED_GRANDPARENTS}
actual_new = {f for f in wm_extended_forward.facts if f[0] == "two_generations_above"}
explain_check("New rule works without engine edits", actual_new, expected_new)

wm_extended_backward = WorkingMemory(DEMO_FACTS)
query = ("two_generations_above", "mary", "sam")
answers = {substitute(query, b)
           for b in backward_chain(wm_extended_backward, extended_rules, query)}
explain_check("Backward chaining supports new predicate", answers, {query})


## 14. Robustness discussion: recursion must terminate

Before calling Task 1 finished, agree how unsuccessful recursive searches terminate. The starter explicitly calls for bounded recursion.

As a separate development check, use a cyclic relationship dataset such as `link(a, b)` and `link(b, a)` with direct and recursive `reachable` rules. Ask for `reachable(a, missing)` and verify that the engine returns no proof without endless recursion. Also verify a reachable answer still succeeds. These generic graph facts avoid interpreting a cycle as a realistic family.

Run this only after you implement a recursion guard, preferably in a separate test process with a timeout. An uncontrolled implementation could otherwise hang the notebook.

A depth limit can be sufficient for the documented bounded demonstration, but may miss longer proofs. Explain that limitation and make the chosen limit cover the longest proof you demonstrate. Do not equate termination with completeness for arbitrary rule bases. A path-based guard also needs care with partially bound goals.

This is an additional robustness check; the main required recursion evidence is the successful ancestry demonstration above.


## 15. Acceptance checklist and presentation sequence

Task 1 is ready when the team can tick all of these:

- [ ] Engine and knowledge are separate; adding the new rule required only data.
- [ ] Forward chaining derives exactly the expected 19 facts.
- [ ] Original facts remain and a second forward run adds nothing.
- [ ] Backward chaining starts from original facts and leaves WM unchanged.
- [ ] True and unsupported ground queries behave correctly.
- [ ] Variable queries return all expected answers, including recursively derived ones.
- [ ] Freshening allows repeated use of the recursive rule.
- [ ] Both directions agree over the same rule base.
- [ ] Recursion termination and its limitations are documented.
- [ ] Restart Kernel → Run All succeeds with your final modules.
- [ ] Each student can explain at least one forward firing and the recursive backward proof.

For a short live presentation, show the fixture and rules, the forward closure, the rerun at fixpoint, one grandparent query, the recursive Mary-to-Leo query, all Mary's descendants, and forward/backward agreement. Keep the other checks available for questions. This is only the Task 1 portion of the assignment's overall 7–10 minute presentation; leave time for Task 2.

### Optional trace, after correctness

A compact trace can show the rule matched, its bindings and conclusion for forward chaining, and the goal, chosen rule and subgoals for backward chaining. Make tracing optional; do not turn engine logic into notebook-specific display code. The trace is useful evidence, but exact firing order and proof order are not acceptance criteria.

### Notes to capture for the report

What representations did we choose? How are conjunctions matched? Why does forward chaining stop? Why are variables freshened in backward chaining? Why is this domain monotonic? Why is there no need to preserve changing world facts across time in Task 1? Which limitation does our recursion strategy introduce?
